# HP-tuning stability check (step 3, `docs/hp_tuning_design.md`)

Tests whether the *timing* of the HP-tuning block matters. For a subset of stocks,
the same tune → freeze → walk-forward-evaluate pipeline runs **four times**, varying
only where the single 10-day tuning block sits in the sample: at **0%, 25%, 50%, or
75%** of the sample period (5 evenly spaced day-pairs each, same protocol as
`nonlinearity.ipynb` Part 1). Each run evaluates on the days **after** its tuning
block — later tuning means fresher HPs but a shorter eval window; the 0% run is
exactly the design of the full sweep.

Per offset the evaluation fits, per split (train day *i* → test day *i+1*),
**XGBoost** with that offset's frozen per-stock-target winners (incl.
`n_estimators` — no early stopping, no test-day contact). No OLS leg: OLS does
not depend on the tuning block, so its per-day results are offset-invariant and
already exist from the full sweep at 0%.

If all four offsets give the same XGB MSE ratios, the timing of tuning is
irrelevant: tuning once at 0% generalizes and the full search only needs to
happen once. The search is checkpointed per offset × stock, so the notebook can
be interrupted and re-run.

Outputs under `model_outputs/XGBoost/Tuning/stability_seed<SEED>/`:
- `trials_off<pct>/<symbol>.parquet` — random-search trials (checkpoints)
- `best_params_off<pct>.json` — frozen winners per offset
- `daily_diagnostics.parquet` — per stock × day × target × offset_pct

In [ ]:
import os, sys, json, warnings
import time
from tqdm.auto import tqdm

import numpy as np
import pandas as pd

from xgboost import XGBRegressor

sys.path.append(os.path.dirname(os.getcwd()))
import importlib
import utils.data_processing as du
import utils.model_utils as mu

importlib.reload(du)
importlib.reload(mu)

In [ ]:
# ============================================================
# Configuration — same tuning protocol as nonlinearity.ipynb Part 1;
# the only new axis is OFFSETS (where the tuning block sits).
# ============================================================

STABILITY_SYMBOLS = du.SYMBOLS[:4]   # subset of stocks for the check

OFFSETS = [0.00, 0.25, 0.50, 0.75]   # tuning-block start as a fraction of the sample

TUNE_BLOCK_LEN = 10                  # trading days per tuning block

N_TRIALS = 40                        # random-search trials per stock-target

N_PAIRS = 5                          # (train, validation) day-pairs per trial

SEED = 0

HORIZONS = ["100ms", "2s", "30s", "5m"]

# Searched hyperparameters: name -> (distribution, low, high)
# "log"/"logint" sample log-uniformly; "int"/"logint" round to integers.
SEARCH_SPACE = {
    "max_depth": ("int", 1, 6),
    "learning_rate": ("log", 0.01, 0.3),
    "min_child_weight": ("logint", 5, 100),
    "subsample": ("uniform", 0.5, 1.0),
    "colsample_bytree": ("uniform", 0.5, 1.0),
    "reg_lambda": ("log", 0.01, 10.0),
}

# Fixed (non-searched) settings for every trial fit; n_estimators is only an
# upper bound — early stopping on the validation day picks the actual tree count.
BASE_PARAMS = dict(
    n_estimators=2000,
    early_stopping_rounds=50,
    eval_metric="rmse",
    tree_method="hist",
    max_bin=128,
    device=mu.select_device(),   # least-used GPU, else "cpu"
    n_jobs=-1,
    random_state=0,
)

# Fixed (non-tuned) settings for the frozen-param evaluation fits.
XGB_PARAMS = dict(
    tree_method="hist",
    max_bin=128,
    device=BASE_PARAMS["device"],
    n_jobs=-1,
    random_state=0,
)
print(f"XGBoost device: {BASE_PARAMS['device']}")

In [ ]:
# ============================================================
# Search functions (copies of nonlinearity.ipynb Part 1)
# ============================================================

def sample_params(rng: np.random.Generator) -> dict:
    """Draw one random-search trial from SEARCH_SPACE."""
    params = {}
    for name, (kind, lo, hi) in SEARCH_SPACE.items():
        if kind == "int":
            params[name] = int(rng.integers(lo, hi + 1))
        elif kind == "uniform":
            params[name] = float(rng.uniform(lo, hi))
        elif kind == "log":
            params[name] = float(np.exp(rng.uniform(np.log(lo), np.log(hi))))
        elif kind == "logint":
            params[name] = int(round(np.exp(rng.uniform(np.log(lo), np.log(hi)))))
        else:
            raise ValueError(f"Unknown distribution kind: {kind}")
    return params


def random_search(
        pairs: list,
        base_params: dict,
        n_trials: int,
        seed: int = 0,
) -> pd.DataFrame:
    """
    Seeded random search for one target (1D y) with day-pair validation.
    Per-pair score = MSE ratio vs. the zero-return benchmark,
    mean(y_val^2) / best_score^2 (scale-invariant); trial score = mean over
    pairs (higher = better). `n_estimators_frozen` = median early-stopped
    best_iteration across the pairs of this config.
    """
    rng = np.random.default_rng(seed)
    rows = []

    for trial in range(n_trials):
        params = sample_params(rng)
        ratios, iterations = [], []

        start = time.perf_counter()
        for X_tr, y_tr, X_val, y_val in pairs:
            model = XGBRegressor(**{**base_params, **params})
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
            mse_bench = float(np.mean(y_val ** 2))
            ratios.append(mse_bench / float(model.best_score) ** 2)
            iterations.append(int(model.best_iteration))

        rows.append({
            "trial": trial,
            **params,
            "mean_mse_ratio": float(np.mean(ratios)),
            "n_estimators_frozen": int(np.median(iterations)),
            "fit_seconds": time.perf_counter() - start,
        })

    return pd.DataFrame(rows)


def freeze_winners(all_trials: pd.DataFrame) -> dict:
    """Winning config per stock-target (highest mean_mse_ratio), with
    n_estimators frozen at that config's median stopped-round."""
    best_rows = all_trials.loc[
        all_trials.groupby(["symbol", "target"])["mean_mse_ratio"].idxmax()
    ]
    best_params = {}
    for _, row in best_rows.iterrows():
        params = {
            name: (int(row[name]) if SEARCH_SPACE[name][0] in ("int", "logint") else float(row[name]))
            for name in SEARCH_SPACE
        }
        params["n_estimators"] = max(1, int(row["n_estimators_frozen"]))
        best_params.setdefault(row["symbol"], {})[row["target"]] = params
    return best_params

In [ ]:
# ============================================================
# Derived setup: one tuning block + eval window per offset
# ============================================================
warnings.filterwarnings("ignore")

PARENT = os.path.dirname(os.getcwd())
DATA_ROOT = f"{PARENT}/data/processed"
STABILITY_DIR = f"{PARENT}/model_outputs/XGBoost/Tuning/stability_seed{SEED}"

sample_cols = pd.read_parquet(
    f"{DATA_ROOT}/{STABILITY_SYMBOLS[0]}/{du.SAMPLE_DATES[0]}.parquet"
).columns
FEATURE_COLS = list(sample_cols[sample_cols.str.startswith("F_")])
TARGET_COLS = [c for c in sample_cols
               if c.startswith("T_") and c.rsplit("_", 1)[-1] in HORIZONS]

# Per offset: block = TUNE_BLOCK_LEN consecutive days starting at
# round(offset * n_days); eval window = all days strictly after the block.
ALL_DATES = list(du.SAMPLE_DATES)
BLOCKS = {}
for off in OFFSETS:
    start = round(off * len(ALL_DATES))
    tune_dates = ALL_DATES[start:start + TUNE_BLOCK_LEN]
    eval_dates = ALL_DATES[start + TUNE_BLOCK_LEN:]
    if len(tune_dates) < TUNE_BLOCK_LEN or len(eval_dates) < 2:
        raise ValueError(f"offset {off:.0%}: not enough days (tune {len(tune_dates)}, eval {len(eval_dates)})")
    pair_idx = sorted(set(
        np.linspace(0, len(tune_dates) - 2, N_PAIRS).round().astype(int)
    ))
    BLOCKS[off] = {"tune_dates": tune_dates, "eval_dates": eval_dates, "pair_idx": pair_idx}
    print(f"{off:>4.0%}: tune {tune_dates[0]} .. {tune_dates[-1]}, "
          f"eval {eval_dates[0]} .. {eval_dates[-1]} ({len(eval_dates)} days)")

In [ ]:
# ============================================================
# Main loop: offset -> tune (checkpointed) -> freeze -> evaluate
# ============================================================
GLOBAL_START = time.perf_counter()
daily_results = []

# Load every symbol's full sample once; tune/eval slice the dates they need.
CACHES = {
    symbol: mu.load_day_cache(DATA_ROOT, symbol, ALL_DATES, FEATURE_COLS, TARGET_COLS)
    for symbol in tqdm(STABILITY_SYMBOLS, desc="load", unit="stock")
}

for off in tqdm(OFFSETS, desc="Offsets", unit="offset"):
    off_pct = int(off * 100)
    block = BLOCKS[off]
    tune_dates, eval_dates, pair_idx = block["tune_dates"], block["eval_dates"], block["pair_idx"]
    trials_dir = f"{STABILITY_DIR}/trials_off{off_pct}"

    # ---- Tuning (identical protocol to nonlinearity.ipynb Part 1,
    #      checkpointed per offset x stock)
    for symbol in tqdm(STABILITY_SYMBOLS, desc=f"tune off{off_pct}", leave=False, unit="stock"):
        if os.path.exists(f"{trials_dir}/{symbol}.parquet"):
            tqdm.write(f"off{off_pct} {symbol}: checkpoint found, skipping")
            continue

        cache = CACHES[symbol]

        symbol_trials = []
        for j, target in enumerate(TARGET_COLS):
            pairs = [
                (cache[tune_dates[i]]["X"], mu.scale_target(cache[tune_dates[i]]["Y"][:, j]),
                 cache[tune_dates[i+1]]["X"], mu.scale_target(cache[tune_dates[i+1]]["Y"][:, j]))
                for i in pair_idx
            ]
            trials = random_search(pairs, base_params=BASE_PARAMS, n_trials=N_TRIALS, seed=SEED)
            trials.insert(0, "target", target)
            trials.insert(0, "symbol", symbol)
            symbol_trials.append(trials)

        mu.save_table(
            df=pd.concat(symbol_trials, ignore_index=True),
            root_dir=trials_dir,
            filename=f"{symbol}.parquet",
        )

    all_trials = pd.concat(
        [pd.read_parquet(f"{trials_dir}/{symbol}.parquet") for symbol in STABILITY_SYMBOLS],
        ignore_index=True,
    )
    best_params = freeze_winners(all_trials)
    with open(f"{STABILITY_DIR}/best_params_off{off_pct}.json", "w") as f:
        json.dump({"offset": off, "tune_dates": tune_dates, "params": best_params}, f, indent=2)

    # ---- Evaluation on the days after the block: frozen-param XGB
    # (no OLS leg — OLS is offset-invariant; its per-day results come from
    #  the full sweep at 0%)
    for symbol in tqdm(STABILITY_SYMBOLS, desc=f"eval off{off_pct}", leave=False, unit="stock"):
        day_cache = CACHES[symbol]
        symbol_params = best_params[symbol]

        for i in range(len(eval_dates) - 1):
            train_day, test_day = eval_dates[i], eval_dates[i + 1]
            X_train, X_test = day_cache[train_day]["X"], day_cache[test_day]["X"]
            Y_train, Y_test = day_cache[train_day]["Y"], day_cache[test_day]["Y"]

            # XGB, frozen params, no early stopping (targets in scaled units)
            xgb_resid = np.empty_like(Y_test)
            for j, target in enumerate(TARGET_COLS):
                model = XGBRegressor(**{**XGB_PARAMS, **symbol_params[target]})
                model.fit(X_train, mu.scale_target(Y_train[:, j]))
                xgb_resid[:, j] = Y_test[:, j] - mu.unscale_prediction(model.predict(X_test))

            rows = mu.daily_diagnostic_rows(
                resid=xgb_resid,
                Y_test=Y_test,
                target_cols=TARGET_COLS,
                train_day=train_day,
                test_day=test_day,
                symbol=symbol,
                run_id=f"stability_off{off_pct}",
                n_train=X_train.shape[0],
                n_test=X_test.shape[0],
            )
            daily_results += [dict(r, model_leg="xgb", offset_pct=off_pct) for r in rows]

tqdm.write(f"TOTAL STABILITY-CHECK TIME: {time.perf_counter()-GLOBAL_START:.2f}s")

mu.save_table(
    df=pd.DataFrame(daily_results),
    root_dir=STABILITY_DIR,
    filename="daily_diagnostics.parquet",
)

In [ ]:
# ============================================================
# Summary: does the tuning-block timing change the conclusion?
# 1) Mean XGB MSE ratio per offset x horizon (higher = better).
# 2) Same comparison on the SHARED eval window (days after the 75% block,
#    present in every run) so offsets are compared on identical days,
#    not windows of different lengths.
# ============================================================
daily_out = pd.read_parquet(f"{STABILITY_DIR}/daily_diagnostics.parquet")
daily_out["horizon"] = daily_out["target"].str.rsplit("_", n=1).str[-1]
daily_out["horizon"] = pd.Categorical(daily_out["horizon"], categories=HORIZONS, ordered=True)

display(
    daily_out.pivot_table(index="horizon", columns="offset_pct",
                          values="mse_ratio", observed=True).round(4)
)

shared = daily_out[daily_out["test_day"].isin(BLOCKS[OFFSETS[-1]]["eval_dates"])]
display(
    shared.pivot_table(index="horizon", columns="offset_pct",
                       values="mse_ratio", observed=True).round(4)
)